In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

# ==========================================================
# النسخة المتوازنة (حل مشكلة الشوائب الصغيرة والكتل العملاقة)
# ==========================================================
def clean_and_count_watershed_balanced(rbc_mask, min_distance=12, min_area=1500, max_area=25000):
    
    # 1. إغلاق الفجوات (خليناه 9x9 عشان نلحم الخلية من غير ما نلزقها في جارتها)
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    closed_mask = cv2.morphologyEx(rbc_mask, cv2.MORPH_CLOSE, kernel_close)

    # 2. سد الخروم الداخلية 
    contours, hierarchy = cv2.findContours(closed_mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    solid_mask = closed_mask.copy()
    if hierarchy is not None:
        for i in range(len(contours)):
            if hierarchy[0][i][3] != -1:
                cv2.drawContours(solid_mask, contours, i, 255, -1)

    # 3. Distance Transform & Blur 
    dist_transform = cv2.distanceTransform(solid_mask, cv2.DIST_L2, 5)
    # قللنا البلور لـ 9x9 عشان القمم تفضل مفصولة
    dist_blurred = cv2.GaussianBlur(dist_transform, (9, 9), 0)

    # 4. إيجاد المراكز (ظبطنا الأرقام للنص عشان نفصل الخلايا الملتصقة)
    local_max_coords = peak_local_max(
        dist_blurred, 
        min_distance=min_distance, 
        threshold_abs=0.8, 
        labels=solid_mask
    )

    markers = np.zeros(solid_mask.shape, dtype=np.int32)
    for i, coord in enumerate(local_max_coords):
        markers[coord[0], coord[1]] = i + 1

    # 5. فصل الخلايا (Watershed)
    labels = watershed(-dist_blurred, markers, mask=solid_mask)

    # 6. استخراج المربعات 
    bboxes = []
    for label_id in np.unique(labels):
        if label_id == 0: 
            continue
        
        cell_mask = np.uint8(labels == label_id) * 255
        cnts, _ = cv2.findContours(cell_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if cnts:
            cnt = max(cnts, key=cv2.contourArea)
            area = cv2.contourArea(cnt)
            
            # السر هنا: 1500 هتموت كل الشوائب اللي تحت، و25000 هتقبل الخلايا السليمة
            if min_area < area < max_area:
                x, y, w, h = cv2.boundingRect(cnt)
                bboxes.append((x, y, w, h))
                    
    return labels, bboxes, solid_mask

# ==========================================
# كود التشغيل
# ==========================================
if 'test_img' in locals() and test_img is not None:
    raw_wbc_mask = get_wbc_mask(test_img)
    clean_wbc = clean_wbc_noise(raw_wbc_mask)
    
    rbc_mask = get_rbc_mask_v3(enhanced_rbc, clean_wbc)
    
    # نستخدم الدالة المتوازنة
    labels, bboxes, solid_mask = clean_and_count_watershed_balanced(rbc_mask)
    
    img_out = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)
    for (x, y, w, h) in bboxes:
        cv2.rectangle(img_out, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cx, cy = x + w//2, y + h//2
        cv2.circle(img_out, (cx, cy), 4, (255, 0, 0), -1)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(enhanced_rbc, cmap="gray")
    axes[0].set_title("1. Enhanced")
    
    axes[1].imshow(solid_mask, cmap="gray")
    axes[1].set_title("2. Solid Mask (Balanced)")
    
    axes[2].imshow(labels, cmap="nipy_spectral")
    axes[2].set_title("3. Watershed Labels")
    
    axes[3].imshow(img_out)
    axes[3].set_title(f"4. Detected RBCs: {len(bboxes)}", color="green")
    
    plt.tight_layout()
    plt.show()

In [ ]:
def get_rbc_mask_v7(enhanced_img, wbc_mask):
    """
    نعالج كل خلية لوحدها بدل ما نعمل closing على الصورة كلها
    """
    # Threshold = 150
    _, binary_mask = cv2.threshold(enhanced_img, 150, 255, cv2.THRESH_BINARY_INV)
    
    # إزالة WBCs
    kernel_dilate = np.ones((7, 7), np.uint8)
    wbc_dilated = cv2.dilate(wbc_mask, kernel_dilate, iterations=2)
    rbc_only = cv2.bitwise_and(binary_mask, cv2.bitwise_not(wbc_dilated))
    
    # Closing صغير بس عشان مايوصلش الخلايا
    kernel_close = np.ones((3, 3), np.uint8)
    rbc_closed = cv2.morphologyEx(rbc_only, cv2.MORPH_CLOSE, kernel_close, iterations=2)
    
    # نلاقي كل contour منفرد
    contours, _ = cv2.findContours(
        rbc_closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    
    filled_mask = np.zeros_like(rbc_closed)
    
    for cnt in contours:
        area = cv2.contourArea(cnt)
        
        # نتجاهل الضوضاء الصغيرة جداً
        if area < 200:
            continue
        
        # نرسم الخلية دي لوحدها في mask مؤقت
        temp = np.zeros_like(rbc_closed)
        cv2.drawContours(temp, [cnt], -1, 255, thickness=cv2.FILLED)
        
        # Flood fill على الخلية دي بس
        h, w = temp.shape
        padded = np.zeros((h+2, w+2), np.uint8)
        padded[1:h+1, 1:w+1] = temp
        
        # نملأ من corner
        flood = padded.copy()
        cv2.floodFill(flood, None, (0, 0), 255)
        flood_inv = cv2.bitwise_not(flood)[1:h+1, 1:w+1]
        
        # نضيف الخلية المملوءة للـ mask النهائي
        filled_mask = cv2.bitwise_or(filled_mask, temp | flood_inv)
    
    # Opening خفيف لإزالة الضوضاء
    kernel_open = np.ones((5, 5), np.uint8)
    rbc_clean = cv2.morphologyEx(filled_mask, cv2.MORPH_OPEN, kernel_open, iterations=1)
    
    return rbc_clean


# ==========================================
# تشغيل
# ==========================================
if test_img is not None:
    raw_wbc_mask = get_wbc_mask(test_img)
    clean_wbc = clean_wbc_noise(raw_wbc_mask)
    
    rbc_mask = get_rbc_mask_v7(enhanced_rbc, clean_wbc)
    
    # Debug: نشوف الـ mask
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(enhanced_rbc, cmap="gray")
    axes[0].set_title("Enhanced")
    axes[1].imshow(rbc_mask, cmap="gray")
    axes[1].set_title("Mask v7 ← لازم دوائر منفصلة")
    
    labels, bboxes = count_rbcs_final(rbc_mask)
    
    img_out = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)
    for (x, y, w, h) in bboxes:
        cv2.rectangle(img_out, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.circle(img_out, (x+w//2, y+h//2), 4, (255, 0, 0), -1)
    
    axes[2].imshow(img_out)
    axes[2].set_title(f"Detected RBCs: {len(bboxes)}", color="green")
    plt.tight_layout()
    plt.show()
    
    print(f"✅ عدد الـ RBCs: {len(bboxes)}")

In [ ]:
def get_rbc_mask_v5(enhanced_img, wbc_mask):
    """
    Otsu محسّن مع CLAHE قبله
    """
    # CLAHE: يحسن التباين محلياً قبل الـ threshold
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    clahe_img = clahe.apply(enhanced_img)
    
    # Otsu بس على الـ CLAHE image
    _, binary_mask = cv2.threshold(
        clahe_img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )
    
    # إزالة WBCs
    kernel_dilate = np.ones((7, 7), np.uint8)
    wbc_mask_dilated = cv2.dilate(wbc_mask, kernel_dilate, iterations=2)
    wbc_mask_inv = cv2.bitwise_not(wbc_mask_dilated)
    rbc_only = cv2.bitwise_and(binary_mask, wbc_mask_inv)
    
    # Closing
    kernel_close = np.ones((5, 5), np.uint8)
    rbc_closed = cv2.morphologyEx(rbc_only, cv2.MORPH_CLOSE, kernel_close, iterations=3)
    
    # Fill rings
    rbc_filled = fill_rbc_rings(rbc_closed)
    
    # Opening
    kernel_open = np.ones((7, 7), np.uint8)
    rbc_clean = cv2.morphologyEx(rbc_filled, cv2.MORPH_OPEN, kernel_open, iterations=1)
    
    return rbc_clean, clahe_img  # نرجع الـ clahe عشان نشوفه


# ==========================================
# تشغيل مع debug للـ mask
# ==========================================
if test_img is not None:
    raw_wbc_mask = get_wbc_mask(test_img)
    clean_wbc = clean_wbc_noise(raw_wbc_mask)
    
    rbc_mask, clahe_img = get_rbc_mask_v5(enhanced_rbc, clean_wbc)
    
    # ← أول حاجة: نشوف الـ mask صح ولا لا
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(enhanced_rbc, cmap="gray")
    axes[0].set_title("Original Enhanced")
    axes[1].imshow(clahe_img, cmap="gray")
    axes[1].set_title("After CLAHE")
    axes[2].imshow(rbc_mask, cmap="gray")
    axes[2].set_title("RBC Mask - هنا المفروض تشوف دوائر بيضاء")
    plt.tight_layout()
    plt.show()
    
    # لو الـ mask صح → نكمل العد
    labels, bboxes = count_rbcs_final(rbc_mask)
    
    img_out = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)
    for (x, y, w, h) in bboxes:
        cv2.rectangle(img_out, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cx, cy = x + w//2, y + h//2
        cv2.circle(img_out, (cx, cy), 4, (255, 0, 0), -1)
    
    fig2, axes2 = plt.subplots(1, 4, figsize=(20, 5))
    axes2[0].imshow(enhanced_rbc, cmap="gray")
    axes2[0].set_title("1. Enhanced")
    axes2[1].imshow(rbc_mask, cmap="gray")
    axes2[1].set_title("2. CLAHE Mask")
    axes2[2].imshow(labels, cmap="nipy_spectral")
    axes2[2].set_title("3. Watershed Labels")
    axes2[3].imshow(img_out)
    axes2[3].set_title(f"4. Detected RBCs: {len(bboxes)}", color="green")
    plt.tight_layout()
    plt.show()